# Rare-cell downsampling benchmark

This notebook runs a controlled rare-cell downsampling benchmark on a processed CITE-seq AnnData object. It compares RNA-only, protein-only, and joint RNA–protein representations by measuring how well a selected target cell type is preserved as it becomes artificially rare.

**Required input:** `data/processed/pbmc5k_representations.h5ad` (or another processed `.h5ad` with PCA embeddings)

**Outputs saved to:** `results/tables/`, `results/metrics/`, `results/logs/`, `results/reports/`


## Setup

The notebook can be run from the repository root or from inside the `notebooks/` directory. Outputs are written under `results/` using the `{output_prefix}__{descriptor}.{ext}` naming convention.


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import anndata as ad

from rarecell.benchmark import (
    TABLES_DIR, INTERMEDIATE_DIR, ensure_project_directories,
    write_markdown_report, validate_representations,
)
from rarecell.downsampling import summarize_downsampling_grid, validate_target_label
from rarecell.naming import make_output_prefix
from rarecell.validation import resolve_label_key, resolve_target_label

ensure_project_directories()

DATA_DIR = PROJECT_ROOT / "data"


## Parameters

Edit these values to change the input file, target cell type, representation keys, downsampling grid, or output prefix.

In [2]:
input_path = DATA_DIR / "processed" / "pbmc5k_representations.h5ad"
preferred_label_key = "cell_type_simple"
preferred_target_label = "B cell"
representation_keys = ["rna_pca", "protein_pca", "joint_pca"]
retain_fractions = [1.0, 0.5, 0.25, 0.1, 0.05]
seeds = [0, 1, 2, 3, 4]
n_neighbors = 15
output_prefix = None


## Load the processed AnnData object

This expects a benchmark-ready object created by the data loading and baseline representation notebooks.

In [3]:
if not input_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {input_path}\n"
        "Run notebooks/data_loading_and_qc.ipynb and "
        "notebooks/baseline_representations.ipynb first, or update input_path."
    )
loaded_input_path = input_path
adata = ad.read_h5ad(input_path)
adata

AnnData object with n_obs × n_vars = 5527 × 21421
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'leiden'
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'original_feature_name', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'protein_names', 'rank_genes_groups', 'umap'
    obsm: 'X_joint_simple', 'X_joint_umap', 'X_pca', 'X_protein_pca', 'X_protein_umap', 'X_rna_pca', 'X_rna_umap', 'X_umap', 'protein_counts'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

## Validate labels and representations

The benchmark requires RNA PCA, protein PCA, one joint RNA-protein representation, and a usable label column.

In [4]:
label_key = resolve_label_key(adata, preferred=preferred_label_key)
target_label = resolve_target_label(adata, label_key, preferred=preferred_target_label)
if output_prefix is None:
    output_prefix = make_output_prefix("pbmc5k", target_label)

validate_target_label(adata, label_key, target_label)
validate_representations(adata, representation_keys)
print(f"Using label key: {label_key!r}")
print(f"Using representation names: {representation_keys}")
print(f"Target label: {target_label!r}")
print(f"Output prefix: {output_prefix!r}")


Using label key: 'leiden'
Using representation names: ['rna_pca', 'protein_pca', 'joint_pca']
Target label: '14'
Output prefix: 'pbmc5k_14'


## Cell-type counts and target candidates

The table below shows the label distribution used by the benchmark. Candidate target populations exclude missing and unknown labels, then prefer populations with enough cells for repeated downsampling.

In [5]:
cell_counts = (
    adata.obs[label_key].astype(str)
    .value_counts().rename_axis("cell_type").reset_index(name="n_cells")
)
cell_counts_path = TABLES_DIR / f"{output_prefix}__cell_type_counts.csv"
cell_counts.to_csv(cell_counts_path, index=False)
display(cell_counts)


,cell_type,n_cells
0,0,952
1,1,730
2,2,699
3,3,600
4,4,590
5,5,395
6,6,383
7,7,345
8,8,176
9,9,175


## Run benchmark helper functions

The benchmark imports shared logic from `src/rarecell/benchmark.py` and `src/rarecell/` (downsampling + metrics).


In [6]:
from rarecell.benchmark import (
    cell_counts_by_fraction,
    make_metric_summary,
    run_downsampling_benchmark,
    save_benchmark_results,
    validate_results_table,
)


## Downsampling grid

Each row defines a retained target-cell fraction and random seed. All non-target cells are retained.

In [7]:
grid = summarize_downsampling_grid(adata, label_key, target_label, retain_fractions, seeds)
grid_path = TABLES_DIR / f"{output_prefix}__downsampling_grid.csv"
grid.to_csv(grid_path, index=False)
display(grid)


,label_key,target_label,retain_fraction,seed,original_n_cells,original_n_target,expected_retained_n_target,expected_n_other,expected_n_cells
0,leiden,14,1.00,0,5527,32,32,5495,5527
1,leiden,14,1.00,1,5527,32,32,5495,5527
2,leiden,14,1.00,2,5527,32,32,5495,5527
3,leiden,14,1.00,3,5527,32,32,5495,5527
4,leiden,14,1.00,4,5527,32,32,5495,5527
5,leiden,14,0.50,0,5527,32,16,5495,5511
6,leiden,14,0.50,1,5527,32,16,5495,5511
7,leiden,14,0.50,2,5527,32,16,5495,5511
8,leiden,14,0.50,3,5527,32,16,5495,5511
9,leiden,14,0.50,4,5527,32,16,5495,5511


## Run the benchmark

For each retained fraction and seed, the target cells are downsampled once and each representation is evaluated on that same downsampled object.

In [8]:
results = run_downsampling_benchmark(
    adata,
    label_key=label_key,
    target_label=target_label,
    representation_keys=representation_keys,
    retain_fractions=retain_fractions,
    seeds=seeds,
    n_neighbors=n_neighbors,
)
print(f"Benchmark complete: {len(results)} rows, {results['representation'].nunique()} representations")
display(results.head())


Benchmark complete: 75 rows, 3 representations


,dataset,target_cell_type,target_label,label_key,fraction,retain_fraction,seed,representation,n_neighbors,n_cells_total,...,n_target_remaining,n_target,n_other,target_fraction,precision,recall,f1,neighborhood_purity,silhouette_target,target_silhouette
0,unknown,14,14,leiden,1.0,1.0,0,rna_pca,15,5527,...,32,32,5495,0.00579,1.0,0.96875,0.984127,0.958333,0.499258,0.499258
1,unknown,14,14,leiden,1.0,1.0,0,protein_pca,15,5527,...,32,32,5495,0.00579,0.8,0.37500,0.510638,0.277083,-0.003456,-0.003456
2,unknown,14,14,leiden,1.0,1.0,0,joint_pca,15,5527,...,32,32,5495,0.00579,1.0,0.96875,0.984127,0.950000,0.317227,0.317227
3,unknown,14,14,leiden,1.0,1.0,1,rna_pca,15,5527,...,32,32,5495,0.00579,1.0,0.96875,0.984127,0.958333,0.499258,0.499258
4,unknown,14,14,leiden,1.0,1.0,1,protein_pca,15,5527,...,32,32,5495,0.00579,0.8,0.37500,0.510638,0.277083,-0.003456,-0.003456


## Validate results and save cell-count metadata

The matching script (`run_downsampling_benchmark.py`) validates the result table
structure and saves per-fraction cell counts to `results/intermediate/`. Both steps
are reproduced here for full parity.

In [9]:
validate_results_table(results, representation_keys, retain_fractions, seeds)
print(f"Results validation passed: {len(results)} rows.")

counts_by_fraction = cell_counts_by_fraction(adata, label_key, target_label, retain_fractions, seeds)
counts_path = INTERMEDIATE_DIR / f"{output_prefix}__cell_counts_by_fraction.csv"
counts_path.parent.mkdir(parents=True, exist_ok=True)
counts_by_fraction.to_csv(counts_path, index=False)
print(f"Saved cell-count metadata: {counts_path}")
display(counts_by_fraction.head())


Results validation passed: 75 rows.
Saved cell-count metadata: /Users/zeyuyao/Documents/GitHub/rarecell-citeseq/results/intermediate/pbmc5k_14__cell_counts_by_fraction.csv


,retain_fraction,seed,cell_type,n_cells,target_label
0,1.0,0,0,952,14
1,1.0,0,1,730,14
2,1.0,0,2,699,14
3,1.0,0,3,600,14
4,1.0,0,4,590,14


## Save result tables

Results are saved via `save_benchmark_results()` which writes CSV and, if `pyarrow` is available, parquet.


In [10]:
save_benchmark_results(results, output_prefix)
results_csv = TABLES_DIR / f"{output_prefix}__benchmark_results.csv"
print(f"Saved: {results_csv}")


Saved: /Users/zeyuyao/Documents/GitHub/rarecell-citeseq/results/tables/pbmc5k_14__benchmark_results.csv


## Metric summary

The summary table groups by representation and retained target-cell fraction, then reports mean and standard deviation across seeds.

In [11]:
metric_summary = make_metric_summary(results)
metric_summary_path = TABLES_DIR / f"{output_prefix}__metric_summary.csv"
metric_summary.to_csv(metric_summary_path, index=False)
display(metric_summary)


,representation,retain_fraction,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,neighborhood_purity_mean,neighborhood_purity_std,target_silhouette_mean,target_silhouette_std,n_target_mean,n_target_std
0,joint_pca,0.05,0.0,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,1.0,0.0
1,joint_pca,0.10,0.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.111111,0.022222,0.206280,0.089841,3.0,0.0
2,joint_pca,0.25,1.0,0.000000,0.87500,0.088388,0.931429,0.050575,0.418333,0.028504,0.446047,0.043344,8.0,0.0
3,joint_pca,0.50,1.0,0.000000,0.95000,0.027951,0.974194,0.014426,0.915000,0.030704,0.431311,0.004844,16.0,0.0
4,joint_pca,1.00,1.0,0.000000,0.96875,0.000000,0.984127,0.000000,0.950000,0.000000,0.317227,0.000000,32.0,0.0
5,protein_pca,0.05,0.0,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,1.0,0.0
6,protein_pca,0.10,0.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.026667,0.028974,-0.018278,0.022763,3.0,0.0
7,protein_pca,0.25,0.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.050000,0.016667,-0.000758,0.018346,8.0,0.0
8,protein_pca,0.50,0.8,0.447214,0.16250,0.168866,0.252871,0.231159,0.159167,0.067340,-0.017664,0.008977,16.0,0.0
9,protein_pca,1.00,0.8,0.000000,0.37500,0.000000,0.510638,0.000000,0.277083,0.000000,-0.003456,0.000000,32.0,0.0


## Benchmark report

A Markdown report is written to `results/reports/` summarising inputs, metrics, and output paths.


In [12]:
report_path = write_markdown_report(
    output_prefix=output_prefix,
    input_file=str(input_path),
    label_key=label_key,
    target_label=target_label,
    representation_keys=representation_keys,
    retain_fractions=retain_fractions,
    seeds=seeds,
    n_neighbors=n_neighbors,
    n_cells=int(adata.n_obs),
    original_target_cells=int((adata.obs[label_key].astype(str) == str(target_label)).sum()),
    n_cell_types=int(adata.obs[label_key].nunique(dropna=False)),
    output_files=[
        str(grid_path),
        str(results_csv),
        str(metric_summary_path),
        str(counts_path),
    ],
    metric_summary=metric_summary,
)
print(f"Report: {report_path}")


Report: /Users/zeyuyao/Documents/GitHub/rarecell-citeseq/results/reports/pbmc5k_14__benchmark_report.md


## Summary

In [13]:
# Summary of generated outputs and deviations from run_downsampling_benchmark.py
generated = [
    cell_counts_path,
    grid_path,
    results_csv,
    metric_summary_path,
    counts_path,
    report_path,
]
print("Generated outputs:")
for p in generated:
    from pathlib import Path as _Path

    p = _Path(p)
    status = "OK" if p.exists() else "MISSING"
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except (ValueError, NameError):
        rel = p
    print(f"  [{status}] {rel}")
print()
print("Deviations from run_downsampling_benchmark.py:")
print("  - Figures generated separately by benchmark_results_and_figures.ipynb.")
print("  - No logs/{prefix}__run_summary.json (script saves this).")
print("  - No file-level logging.")


Generated outputs:
  [OK] results/tables/pbmc5k_14__cell_type_counts.csv
  [OK] results/tables/pbmc5k_14__downsampling_grid.csv
  [OK] results/tables/pbmc5k_14__benchmark_results.csv
  [OK] results/tables/pbmc5k_14__metric_summary.csv
  [OK] results/intermediate/pbmc5k_14__cell_counts_by_fraction.csv
  [OK] results/reports/pbmc5k_14__benchmark_report.md

Deviations from run_downsampling_benchmark.py:
  - Figures generated separately by benchmark_results_and_figures.ipynb.
  - No logs/{prefix}__run_summary.json (script saves this).
  - No file-level logging.
